# Kurs Kassası: qəbz oxuyucusu - QLoRA + vLLM (addım 5)

Valideyn kursa pul atanda rəhbərə bank SMS-i, kağız qəbzin şəkli və ya «Ayselin noyabr pulunu
köçürdüm» kimi mesaj gəlir. Kiçik model (**Qwen2.5-1.5B-Instruct**) bu mətndən JSON çıxarır:
məbləğ, tarix, ödəyən, kartın son 4 rəqəmi, hansı ay, şagird (adlıq halda).

**Niyə öz modelimiz:** mətndə ad və kart rəqəmləri var. Groq/Claude-a göndərmək əvəzinə öz GPU-muzda
işləyən kiçik model (`RECEIPT_ALLOW_EXTERNAL_LLM=false` defoltdur).

**Plan:** sintetik dataset → 4-bit baza → LoRA → öyrətmə → baza vs fine-tune vs qaydalar
(görülən və **görülməyən** mətn növlərində) → vLLM multi-LoRA serve → JSON məcburi dekodlaşdırma →
sürət → MasterAI-yə qoşmaq. Runtime: **T4 GPU**. Data sintetikdir (`ml/receipts/synth.py`), heç bir
real qəbz yüklənmir.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Paketlər
Versiyalar `ml/qlora_mcq.ipynb` ilə eynidir: TRL/PEFT API-ləri tez dəyişir.

In [ ]:
!pip -q install "transformers==4.46.3" "peft==0.13.2" "trl==0.12.2" "bitsandbytes==0.45.0" \
    "datasets==3.1.0" "accelerate==1.1.1"
!git clone -q https://github.com/Alihuseyn-ibrahimov/masterai.git
!pip -q install pydantic

## 2. Dataset (Colab-da yaranır)
6 mətn növü. `ewallet` (komissiya məbləği çaşdırıcıdır) və `messenger` (qeyri-standart sahə adları)
öyrətməyə DÜŞMÜR: `test_unseen` modelin görmədiyi formatda ümumiləşdirib-ümumiləşdirmədiyini ölçür.
Komanda qaydalı bazanın nəticəsini də çap edir.

In [ ]:
!cd masterai && python -m ml.receipts.sft --per-family 400 --seed 7 --out ../receipts
import json, sys
sys.path.insert(0, "masterai")

def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

rows = {name: read_jsonl(f"receipts/{name}.jsonl") for name in ("train", "val", "test_seen", "test_unseen")}
print({k: len(v) for k, v in rows.items()})
print(rows["train"][0]["messages"][1]["content"])
print(rows["train"][0]["messages"][2]["content"])

## 3. Baza model 4-bit (QLoRA)
1.5B model NF4 ilə ~1.2 GB VRAM. Mətnlər qısadır (~150 token), ona görə öyrətmə T4-də ~10 dəqiqədir.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "masterai-receipts-qlora"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4 bf16 dəstəkləmir
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

## 4. Öyrətmə
Sistem və istifadəçi mesajı produksiya ilə bayt-bayt eynidir (`pipeline.receipt_reader.SYSTEM_PROMPT`,
`user_prompt`): modelin öyrəndiyi format serverin göndərdiyi formatdır.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([{"messages": r["messages"]} for r in rows["train"]])
val_ds = Dataset.from_list([{"messages": r["messages"]} for r in rows["val"]])

config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,      # effektiv batch 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_seq_length=768,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
)
trainer = SFTTrainer(model=model, args=config, train_dataset=train_ds, eval_dataset=val_ds,
                     peft_config=lora, processing_class=tokenizer)
trainer.model.print_trainable_parameters()
trainer.train()
trainer.save_model(OUTPUT_DIR)   # yalnız adapter (~35 MB): vLLM onu baza modelin üstünə qoyur

## 5. Baza vs fine-tune vs qaydalar
Hər test dəstindən 200 nümunə, greedy dekodlaşdırma. `ml/receipts/score.py`: sahə üzrə dəqiqlik,
«bütün sahələr düz» və JSON-un oxunma faizi. Əsas sual: `test_unseen`-də QLoRA qaydalardan yaxşıdırmı?

In [ ]:
model = trainer.model
model.eval()
tokenizer.padding_side = "left"

def generate_batch(batch, max_new_tokens=160):
    prompts = [tokenizer.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
               for r in batch]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def run(split, path, use_adapter, limit=200, batch_size=16):
    data = rows[split][:limit]
    with open(f"receipts/{split}_gold200.jsonl", "w", encoding="utf-8") as fh:
        for r in data:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    outputs = []
    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]
        if use_adapter:
            outputs += generate_batch(batch)
        else:
            with model.disable_adapter():
                outputs += generate_batch(batch)
    with open(path, "w", encoding="utf-8") as fh:
        for text in outputs:
            fh.write(json.dumps({"output": text}, ensure_ascii=False) + "\n")

for split in ("test_seen", "test_unseen"):
    run(split, f"receipts/{split}_base.jsonl", use_adapter=False)
    run(split, f"receipts/{split}_qlora.jsonl", use_adapter=True)

In [ ]:
!cd masterai && for s in test_seen test_unseen; do echo "== $s"; python -m ml.receipts.score \
    --gold ../receipts/${s}_gold200.jsonl --rules --pred ../receipts/${s}_base.jsonl --pred ../receipts/${s}_qlora.jsonl; done

## 6. Adapteri saxla
Multi-LoRA serve üçün birləşdirmək LAZIM DEYİL: vLLM eyni baza modelin üstündə bir neçə adapteri
(məs. `masterai-mcq` və `receipts`) birlikdə saxlayır. Drive-a köçür ki, sessiya bitəndə itməsin.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r {OUTPUT_DIR} /content/drive/MyDrive/ && du -sh {OUTPUT_DIR}

## 7. vLLM: multi-LoRA serve
**Runtime → Restart session** et (öyrətmə modeli GPU yaddaşını tutur, vLLM öz torch versiyasını qoyur),
sonra bu hissəni işlət. `--enable-lora --lora-modules receipts=...`: sorğuda `"model": "receipts"`
yazılanda adapter, `"model": "Qwen/Qwen2.5-1.5B-Instruct"` yazılanda baza model cavab verir.

In [ ]:
!pip -q install "vllm==0.6.6.post1"
import os, secrets, subprocess, time, urllib.request

ADAPTER = "/content/masterai-receipts-qlora"
if not os.path.exists(ADAPTER):
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER = "/content/drive/MyDrive/masterai-receipts-qlora"
API_KEY = secrets.token_urlsafe(24)
log = open("vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct", "--dtype", "half", "--max-model-len", "2048",
    "--enable-lora", "--lora-modules", f"receipts={ADAPTER}", "--max-lora-rank", "16",
    "--gpu-memory-utilization", "0.85", "--api-key", API_KEY, "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("vLLM hazırdır"); break
    except Exception:
        time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

## 8. Məcburi JSON (`guided_json`)
vLLM dekodlaşdırmanı JSON sxeminə bağlaya bilir: model sxemə uymayan token seçə bilmir. Fine-tune
ilə birlikdə JSON oxunma faizi 100%-ə çatmalıdır, sual isə **məzmunun** düzgünlüyüdür. Sxem
produksiyadakı `Receipt` pydantic modelindəndir.

In [ ]:
import json, sys, httpx
sys.path.insert(0, "masterai")
from pipeline.receipt_reader import Receipt

SCHEMA = Receipt.model_json_schema()

def ask(row, model="receipts", guided=False):
    body = {"model": model, "max_tokens": 160, "temperature": 0, "messages": row["messages"][:2]}
    if guided:
        body["guided_json"] = SCHEMA
    r = httpx.post("http://localhost:8000/v1/chat/completions", json=body, timeout=120,
                   headers={"Authorization": f"Bearer {API_KEY}"})
    return r.json()["choices"][0]["message"]["content"]

gold = [json.loads(l) for l in open("receipts/test_unseen_gold200.jsonl", encoding="utf-8")]
for name, kwargs in {"vllm_free": {}, "vllm_guided": {"guided": True},
                     "vllm_base_guided": {"guided": True, "model": "Qwen/Qwen2.5-1.5B-Instruct"}}.items():
    with open(f"receipts/{name}.jsonl", "w", encoding="utf-8") as fh:
        for row in gold:
            fh.write(json.dumps({"output": ask(row, **kwargs)}, ensure_ascii=False) + "\n")

In [ ]:
!cd masterai && python -m ml.receipts.score --gold ../receipts/test_unseen_gold200.jsonl \
    --pred ../receipts/vllm_free.jsonl --pred ../receipts/vllm_guided.jsonl --pred ../receipts/vllm_base_guided.jsonl

## 9. Sürət
Qəbz qısadır: əsas göstərici gecikmədir (TTFT, p95), ötürmə qabiliyyəti yox. Eyni test promptları
`ml/bench_llm.py` ilə 1/4/8 paralel.

In [ ]:
!cd masterai && python -m ml.bench_llm --url http://localhost:8000 --model receipts --api-key {API_KEY} \
    --val ../receipts/test_unseen.jsonl --max-tokens 160 --concurrency 1,4,8 --requests 32 --out ../receipts_bench.json

## 10. MasterAI serverinə qoş
Cloudflare tuneli (bax `ml/vllm_serve.ipynb`). Colab bağlananda ünvan ölür: server qaydalı oxuyucuya
düşür, Kassa işləməyə davam edir.

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stdout.readline())
    if match:
        url = match.group(0); break
print("Serverdə /opt/masterai/.env-ə yaz və `systemctl restart masterai`:\n")
print(f"VLLM_BASE_URL={url}\nVLLM_API_KEY={API_KEY}\nVLLM_RECEIPT_MODEL=receipts")